# Results on the Fitzpatrick17k test set

Collects all finished runs and builds the result tables: per run, averaged over seeds, and with bootstrap intervals. Run this after the condition notebooks (01 to 06). No GPU is needed. The tables are saved in `summary/`.

In [1]:
import sys
from pathlib import Path

# The shared code is in the repository root. Look for it from the folder of this notebook upwards.
REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis.py").is_file())
sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import analysis as A

SUMMARY_DIR = A.SUMMARY_DIR
SUMMARY_DIR.mkdir(exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)

runs = A.finished_runs()
print(runs.groupby("condition").seed.apply(sorted))

condition
baseline                        [42, 43, 44]
combined                        [42, 43, 44]
reweighting_effective_number    [42, 43, 44]
reweighting_kamiran             [42, 43, 44]
targeted_colour                 [42, 43, 44]
universal_colour                [42, 43, 44]
Name: seed, dtype: object


## Results per run

In [2]:
per_run = A.per_run_table(runs)
per_run.to_csv(SUMMARY_DIR / "per_run_results.csv", index=False)
per_run

,condition,seed,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_max_min,auc_gap_1_2_minus_5_6,balanced_accuracy_0.5,balanced_accuracy_youden,recall_youden,specificity_youden,youden_threshold,stage2_best_epoch,val_auc
0,baseline,42,0.9193,0.8984,0.9372,0.9527,0.0543,-0.0543,0.8364,0.8380,0.8333,0.8426,0.5480,31,0.9222
1,baseline,43,0.9148,0.9015,0.9320,0.9301,0.0305,-0.0286,0.8380,0.8364,0.8735,0.7994,0.1833,33,0.9318
2,baseline,44,0.9161,0.9013,0.9353,0.9247,0.0340,-0.0234,0.8410,0.8426,0.8519,0.8333,0.2744,21,0.9256
3,universal_colour,42,0.9166,0.8988,0.9408,0.9312,0.0420,-0.0324,0.8364,0.8364,0.8580,0.8148,0.2657,40,0.9336
4,universal_colour,43,0.9221,0.9053,0.9419,0.9312,0.0366,-0.0259,0.8333,0.8302,0.8858,0.7747,0.0805,39,0.9293
5,universal_colour,44,0.9157,0.8984,0.9369,0.9280,0.0385,-0.0296,0.8364,0.8287,0.8735,0.7840,0.2257,28,0.9272
6,targeted_colour,42,0.9211,0.8999,0.9485,0.9280,0.0487,-0.0281,0.8503,0.8441,0.8611,0.8272,0.3904,31,0.9265
7,targeted_colour,43,0.9000,0.8739,0.9254,0.9247,0.0515,-0.0508,0.8194,0.8164,0.8642,0.7685,0.1415,6,0.9184
8,targeted_colour,44,0.9156,0.8975,0.9347,0.9452,0.0476,-0.0476,0.8256,0.8395,0.8549,0.8241,0.2139,37,0.9171
9,reweighting_effective_number,42,0.9141,0.8995,0.9333,0.9237,0.0338,-0.0242,0.8349,0.8380,0.8241,0.8519,0.3758,18,0.9226


## Mean and standard deviation over the seeds

In [3]:
seed_summary = A.seed_summary(per_run)
seed_summary.to_csv(SUMMARY_DIR / "seed_summary.csv")
seed_summary[["n_seeds", "auc_mean", "auc_sd", "auc_fst_1_2_mean", "auc_fst_3_4_mean",
              "auc_fst_5_6_mean", "auc_gap_1_2_minus_5_6_mean", "auc_gap_1_2_minus_5_6_sd"]]

,n_seeds,auc_mean,auc_sd,auc_fst_1_2_mean,auc_fst_3_4_mean,auc_fst_5_6_mean,auc_gap_1_2_minus_5_6_mean,auc_gap_1_2_minus_5_6_sd
condition,,,,,,,,
baseline,3,0.9167,0.0023,0.9004,0.9348,0.9358,-0.0354,0.0165
universal_colour,3,0.9181,0.0035,0.9008,0.9399,0.9301,-0.0293,0.0033
targeted_colour,3,0.9122,0.0109,0.8904,0.9362,0.9326,-0.0422,0.0123
reweighting_effective_number,3,0.9171,0.0026,0.9006,0.9382,0.9262,-0.0256,0.0188
reweighting_kamiran,3,0.9115,0.0058,0.8942,0.9296,0.9312,-0.0370,0.0084
combined,3,0.9170,0.0062,0.8991,0.9387,0.9400,-0.0409,0.0092


## Bootstrap intervals

Seed-averaged AUC with 95% intervals from 2,000 stratified bootstrap resamples of the test set, and the paired difference of every intervention with the baseline. This takes a few minutes.

In [4]:
estimates, differences = A.bootstrap_over_seeds(runs, n_boot=2000)
estimates.to_csv(SUMMARY_DIR / "bootstrap_auc.csv", index=False)
differences.to_csv(SUMMARY_DIR / "bootstrap_diff_vs_baseline.csv", index=False)
table = A.format_ci_table(estimates)
table.to_csv(SUMMARY_DIR / "bootstrap_auc_table.csv", index=False)
table

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,baseline,"0.917 [0.896, 0.936]","0.900 [0.868, 0.931]","0.935 [0.905, 0.961]","0.936 [0.871, 0.980]","-0.035 [-0.092, 0.037]"
1,universal_colour,"0.918 [0.897, 0.937]","0.901 [0.869, 0.930]","0.940 [0.911, 0.965]","0.930 [0.861, 0.978]","-0.029 [-0.088, 0.046]"
2,targeted_colour,"0.912 [0.891, 0.933]","0.890 [0.859, 0.922]","0.936 [0.906, 0.962]","0.933 [0.867, 0.980]","-0.042 [-0.101, 0.036]"
3,reweighting_effective_number,"0.917 [0.896, 0.937]","0.901 [0.869, 0.932]","0.938 [0.909, 0.963]","0.926 [0.851, 0.979]","-0.026 [-0.088, 0.057]"
4,reweighting_kamiran,"0.912 [0.890, 0.932]","0.894 [0.863, 0.925]","0.930 [0.898, 0.957]","0.931 [0.866, 0.976]","-0.037 [-0.094, 0.036]"
5,combined,"0.917 [0.896, 0.937]","0.899 [0.867, 0.930]","0.939 [0.909, 0.963]","0.940 [0.872, 0.986]","-0.041 [-0.100, 0.034]"


In [5]:
# Paired differences from the baseline. A star means that the 95% interval excludes zero.
A.format_diff_table(differences)

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,universal_colour,"+0.001 [-0.006, +0.009]","+0.000 [-0.011, +0.011]","+0.005 [-0.006, +0.017]","-0.006 [-0.036, +0.019]","+0.006 [-0.021, +0.039]"
1,targeted_colour,"-0.004 [-0.011, +0.002]","-0.010 [-0.021, +0.002]","+0.001 [-0.007, +0.010]","-0.003 [-0.027, +0.019]","-0.007 [-0.031, +0.018]"
2,reweighting_effective_number,"+0.000 [-0.006, +0.006]","+0.000 [-0.010, +0.010]","+0.003 [-0.005, +0.013]","-0.010 [-0.040, +0.015]","+0.010 [-0.016, +0.042]"
3,reweighting_kamiran,"-0.005 [-0.011, +0.001]","-0.006 [-0.015, +0.003]","-0.005 [-0.015, +0.004]","-0.005 [-0.021, +0.013]","-0.002 [-0.019, +0.017]"
4,combined,"+0.000 [-0.007, +0.007]","-0.001 [-0.013, +0.010]","+0.004 [-0.006, +0.014]","+0.004 [-0.025, +0.033]","-0.005 [-0.035, +0.026]"
